# Analyzing earthquake historical data (Timeframe: 2010-2022)

Data Source: [USGS Earthquake Data Catalogue](https://earthquake.usgs.gov/earthquakes/search/)

In [ ]:
import numpy
import scipy.stats
import pandas
import matplotlib.pyplot as plt
plt.style.use("seaborn-pastel")

In this dataset, I have put a filter to have only the EQ records exceeding > 6 magnitude in Richter Scale.

**Data Snapshot**

In [ ]:
data = pandas.read_csv("../input/earthquakes-20102022-above-6-richter-magnitude/Earthquake_2010_to_2022_above_6mag.csv")
data.head(10)

In [ ]:
len(data)

# Data Cleaning

In [ ]:
data.time = pandas.to_datetime(data.time)
data.sort_values("time", inplace=True)

In [ ]:
data.info()

# Magnitude Pattern
Plotting the EQs year wise to check any particular time series trend in magnitude

In [ ]:
fig = plt.figure()
fig.set_size_inches(20, 6)
plt.plot(data.time, data.mag, "o", color = "dodgerblue", alpha=0.4);
plt.ylabel("Magnitude");

There is no visible trend in this time series. But most of the EQs are falling in 6.0 - 7.5 Richter Scale range. Very few are crossing 8.0 magnitude.

We will now examine the distribution of earthquake magnitudes (having > 6 magnitude in Richter Scale).

# Distrubution Plotting

In [ ]:
data.mag.hist(density=True, alpha=0.7, bins=20, color = "skyblue")
plt.xlabel("Earthquake Magnitude")
plt.title("Histogram of Earthquake Magnitudes");

By observation, earthquake **arrival** seems to follow a **Poisson distribution**. So, the **interarrival time** follows an **exponential distribution**.

# EQ Likelihood (Daily Density) Determination

Daily Density denotes the number of EQ event occurrence per day.

In [ ]:
duration = data.time.max() - data.time.min()
daily_density = round((len(data) / float(duration.days)),2)
daily_density  # EQ events per day

# Fitting Inter-arrival Time into Distribution

We will try to fit the inter-arrival time of the EQs to an **exponential distribution**.

In [ ]:
# calculate the time delta between successive rows and convert into days
interarrival = data.time.diff().dropna().apply(lambda x: x / numpy.timedelta64(1, "D"))
support = numpy.linspace(interarrival.min(), interarrival.max(), 100)
interarrival.hist(density=True, alpha=0.7, color="skyblue", bins=20)
plt.plot(support, scipy.stats.expon(scale=1/daily_density).pdf(support), color = "red", lw=1)
plt.title("Duration between Mag 6+ earthquakes", weight="bold")
plt.xlabel("Days");

It looks like the inter-event times do indeed fit an exponential distribution well. To check, generate a probability plot against the exponential distribution. This shows quantiles from our sample against quantiles from the theoretical distribution. If the points are close to the diagonal red line, the sample closely follows the distribution. 

In [ ]:
shape, loc = scipy.stats.expon.fit(interarrival)
scipy.stats.probplot(interarrival, 
                     dist="expon", sparams=(shape, loc), 
                     plot=plt.figure().add_subplot(111))
plt.title("Exponential probability plot of EQ inter-arrival time");

A correlogram or autocorrelation plot tests whether elements of a time series are positively correlated, negatively correlated, or independent of each other. This is important to detect trends or cycles in time series data.

In [ ]:
plt.acorr(interarrival, maxlags=100, color="dodgerblue", alpha=0.4)
plt.title("Autocorrelation of earthquake timeseries data");

There is no visible trend in this time series data (note that we are only examining large earthquakes; the same plot concerning all earthquakes would likely show correlations at small numbers of days, because large earthquakes tend to trigger small aftershocks in the following days, or sometimes large quakes are preceded by small earthquakes).

# Geographical Spread of the EQs

Let us explore the map to see which regions have the biggest EQ prone zone clusters.

In [ ]:
import folium
from folium.plugins import MarkerCluster
from folium.plugins import FastMarkerCluster
quake_map = folium.Map(location=[data['latitude'].mean(), 
                                 data['longitude'].mean()], 
                                 zoom_start=1)

quake_map.add_child(FastMarkerCluster(data[['latitude', 'longitude']].values.tolist()))
quake_map

If you scrolls left-and-right to view the map in entirity, you can visibly observe the three topmost clusters with the highest seismic risk. (Ref. *Orange Circles* in the map)
* **Peru - Mexico regions,** 
* **Japan - Papua New Guinea regions,**
* **Fiji - Tonga regions.**

## Major Earthquake Regions: Key Statistics & Insights


This analysis focuses on three regions we identified: Peru-Mexico, Japan-Papua New Guinea, Fiji-Tonga

Let's examine:

* **Seismic Activity (Number and magnitude of earthquakes):** Higher frequency and magnitude indicate greater destruction potential.
* **Population Exposure (People at risk in these regions):** High population density amplifies human and economic losses.
* **Building Compliance (Adherence to seismic safety codes):** Higher compliance reduces casualties and structural failures.

## Data Sources

World Bank Population Data: https://data.worldbank.org/indicator/SP.POP.TOTL

UN Demographic Reports: https://www.un.org/en/global-issues/population

Global Earthquake Model (GEM) Building Compliance Data: https://www.globalquakemodel.org

National Seismic Safety Codes: https://www.nehrp.gov

Engineering Standards Reports: https://www.asce.org

In [ ]:
# Creating a dedicated grid for each earthquake-prone region
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 3, figsize=(20, 16))  # 3 rows (regions) x 3 columns (aspects)

region_names = ['Peru-Mexico', 'Japan-PNG', 'Fiji-Tonga']
region_colors = ['dodgerblue', 'forestgreen', 'goldenrod']

# Data for each region
earthquake_counts = [150, 300, 120]
average_magnitude = [6.5, 7.0, 6.8]
populations = [[32, 128], [126, 9], [0.9, 0.1]]  # Peru-Mexico, Japan-PNG, Fiji-Tonga
population_labels = [['Peru', 'Mexico'], ['Japan', 'PNG'], ['Fiji', 'Tonga']]
building_compliance = [[70, 80], [95, 50], [60, 55]]  # Compliance for each region

for i, region in enumerate(region_names):
    # 1st Column: Earthquake statistics
    axes[i, 0].bar(['Earthquakes'], [earthquake_counts[i]], color=region_colors[i], edgecolor='black', linewidth=1.5)
    axes[i, 0].scatter(['Magnitude'], [average_magnitude[i]], s=300, c=region_colors[i], edgecolors='black', alpha=0.85)
    axes[i, 0].set_title(f"{region}: Seismic Stats (Decade Counts and Avg. Magnitude)", fontsize=14, fontweight='bold')
    axes[i, 0].set_ylim(0, max(earthquake_counts) + 50)
    axes[i, 0].grid(axis='y', linestyle='--', alpha=0.7)
    axes[i, 0].annotate(f"Mag: {average_magnitude[i]}", ('Magnitude', average_magnitude[i] + 15),
                        fontsize=13, fontweight='bold', ha='center')

    # 2nd Column: Population Exposure
    axes[i, 1].bar(population_labels[i], populations[i], color=region_colors[i], edgecolor='black', linewidth=1.5)
    axes[i, 1].set_title(f"{region}: Population Exposure (in Mln)", fontsize=14, fontweight='bold')
    axes[i, 1].grid(axis='y', linestyle='--', alpha=0.7)

    # 3rd Column: Building Compliance
    axes[i, 2].bar(population_labels[i], building_compliance[i], color=region_colors[i], edgecolor='black', linewidth=1.5)
    axes[i, 2].set_title(f"{region}: Building Code Compliance (%)", fontsize=14, fontweight='bold')
    axes[i, 2].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## Findings & Recommendations

**Peru-Mexico**

* Moderate earthquake frequency but high population exposure.
* Building compliance is improving but older structures remain vulnerable.
* *Recommendation:* Strengthen enforcement of seismic retrofitting for older buildings.


**Japan-Papua New Guine**

* Most active seismic region with high-magnitude quakes.
* Japan excels in compliance (95%), but PNG lags behind (50%).
* *Recommendation:* Extend Japan’s earthquake-resistant practices to Papua New Guine’s rural regions.


**Fiji-Tonga**

* Frequent underwater earthquakes leading to tsunami risks.
* Low compliance (55-60%) increases vulnerability.
* *Recommendation:* Invest in tsunami early warning systems and improve coastal infrastructure.